# 随机森林基线模型

直接复用 data/xgboost_features 中已保存的训练、验证、测试特征，与当前 code/XGBOOST.ipynb 使用相同输入。

- 不读取原始数据，不重新构造特征、划分样本或编码。
- 标签 isDefault：1为违约，0为正常；分类阈值固定为0.5。
- 只训练一个基线模型，不调参，不生成分析报告。
- 运行末尾单元格导出 results/RandomForest/metrics.csv，供三模型比较使用。
- 请先运行三个模型并导出新指标，再运行 ModelComparison.ipynb；旧 CSV 不代表本次结果。

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline

project_dir = Path("D:/credit")
feature_dir = project_dir / "data" / "xgboost_features"
result_dir = project_dir / "results" / "RandomForest"
result_dir.mkdir(parents=True, exist_ok=True)
random_state = 42
threshold = 0.5

## 1. 读取已保存的特征

从 metadata.json 读取标签名和特征顺序，直接加载 train.parquet、val.parquet、test.parquet。

In [ ]:
metadata = json.loads((feature_dir / "metadata.json").read_text(encoding="utf-8"))
target_column = metadata["target"]
feature_columns = metadata["feature_columns"]

def load_dataset(split_name):
    data = pd.read_parquet(feature_dir / f"{split_name}.parquet")
    return data.loc[:, feature_columns], data[target_column]

X_train, y_train = load_dataset("train")
X_val, y_val = load_dataset("val")
X_test, y_test = load_dataset("test")

## 2. 样本概况

沿用已保存的样本划分，展示各样本的记录数、违约率和特征数量。

In [ ]:
sample_summary = pd.DataFrame({
    "样本数": [len(y_train), len(y_val), len(y_test)],
    "违约率": [y_train.mean(), y_val.mean(), y_test.mean()],
    "特征数": len(feature_columns),
}, index=["训练集", "验证集", "测试集"]).rename_axis("样本")
display(sample_summary)

## 3. 缺失值填补

只在训练集拟合中位数填补器，验证集、测试集使用同一套转换。
保留已有特征列，不重复编码、筛选或标准化。

In [ ]:
preprocessor = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
).set_output(transform="pandas")

X_train_ready = preprocessor.fit_transform(X_train)
X_val_ready = preprocessor.transform(X_val)
X_test_ready = preprocessor.transform(X_test)

## 4. 训练随机森林基线

100棵树，最大深度10，叶节点最少20条样本，随机种子42。只训练一次，不搜索参数。

In [ ]:
model = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=20,
    random_state=random_state, n_jobs=-1,
)
model.fit(X_train_ready, y_train)

## 5. 主要评估指标

分别计算训练、验证、测试样本的 AUC、KS、Precision、Recall、F1。
分类指标统一使用0.5阈值；表格显示四位小数，导出保留完整数值精度。

In [ ]:
datasets = {
    "训练集": (X_train_ready, y_train),
    "验证集": (X_val_ready, y_val),
    "测试集": (X_test_ready, y_test),
}
metric_rows = []
probabilities = {}
for name, (features, target) in datasets.items():
    probability = model.predict_proba(features)[:, 1]
    probabilities[name] = probability
    prediction = (probability >= threshold).astype(int)
    false_positive_rate, true_positive_rate, _ = roc_curve(target, probability)
    metric_rows.append({
        "样本": name,
        "AUC": roc_auc_score(target, probability),
        "KS": (true_positive_rate - false_positive_rate).max(),
        "Precision": precision_score(target, prediction),
        "Recall": recall_score(target, prediction),
        "F1": f1_score(target, prediction),
    })

metrics = pd.DataFrame(metric_rows).set_index("样本")
display(metrics.round(4))

## 6. 测试集混淆矩阵与 ROC 曲线

混淆矩阵的行为实际标签，列为预测标签。结果只在笔记本中展示，不生成报告或保存图片。

In [ ]:
test_probability = probabilities["测试集"]
test_prediction = (test_probability >= threshold).astype(int)
confusion = pd.DataFrame(
    confusion_matrix(y_test, test_prediction, labels=[0, 1]),
    index=["实际正常", "实际违约"], columns=["预测正常", "预测违约"],
).rename_axis("真实标签")
display(confusion)

false_positive_rate, true_positive_rate, _ = roc_curve(y_test, test_probability)
figure, axis = plt.subplots(figsize=(6, 5))
axis.plot(
    false_positive_rate, true_positive_rate,
    label=f"Test AUC = {metrics.loc['测试集', 'AUC']:.4f}",
)
axis.plot([0, 1], [0, 1], linestyle="--", color="grey")
axis.set(xlabel="False positive rate", ylabel="True positive rate", title="Random forest: Test ROC")
axis.legend()
figure.tight_layout()
plt.show()

## 7. 特征重要性

按加权不纯度下降展示前15个特征，重要性不表示影响方向。

In [ ]:
importance = pd.Series(
    model.feature_importances_, index=X_train_ready.columns, name="重要性",
).sort_values(ascending=False).rename_axis("特征")
top_features = importance.head(15).to_frame()
display(top_features)

## 8. 导出比较指标

仅导出本次运行的 metrics.csv，不生成模型分析报告。三个模型均导出新指标后，再运行比较笔记本。

In [ ]:
export_metrics = metrics.assign(
    样本数=[len(y_train), len(y_val), len(y_test)],
    特征数=len(feature_columns), 阈值=threshold,
    结果来源="当前 Parquet 特征，本次运行",
).reset_index()
export_metrics.to_csv(result_dir / "metrics.csv", index=False, encoding="utf-8-sig")
display(export_metrics)